## Issue 116 (Purav): PETai activity label release, audit and HuggingFace publication

Lead     : `Purav Gupta`

Issue    : [Github Issue #116](https://github.com/petadex/igem-toronto/issues/116) — _publish the PETai activity label set_

Start    : `2026-10-15`

Complete : `2026-10-15`

Files    : `resources/261015_issue116_petai_label_release/` — audit and push scripts (committed); release itself at `/project/6122149/purav/petai-labels/combined/` on Killarney (git-ignored)

HF repo  : [`petadex/petai-activity-labels`](https://huggingface.co/datasets/petadex/petai-activity-labels) — public, four configs

---

### Data Accessed: 2026-10-15
The label release was built on 2026-10-14 and lived only on a laptop. Copied to the cluster with `scp`, unpacked into its own directory, and the macOS resource-fork junk removed.

```bash
# from the laptop
scp ~/Downloads/combined.zip purav@killarney.alliancecan.ca:/project/6122149/purav/

# on Killarney
mkdir -p /project/6122149/purav/petai-labels
unzip -q /project/6122149/purav/combined.zip -d /project/6122149/purav/petai-labels/
rm -rf /project/6122149/purav/petai-labels/__MACOSX
```

Six files: `labels.csv`, `sequences.csv`, `split.csv`, `clusters.csv`, `sequences.fasta`, `summary.json`. 2.9 MB unpacked.

## Introduction

PETadex gives us 307,155,746 catalytic ORFs and no activity labels. Every config of `petadex/petadex-corpus-full` and all 57 columns of `petadex/petadex-corpus-centroids90` were searched and no label column exists in either; `plasticome` is a four-column MD5 identity bridge rather than a label table. Supervised training therefore depends entirely on a separate curated set.

That set is the PETai release: 13,092 measured activity and stability labels on 1,219 sequences, drawn from 696 published references plus the iGEM Toronto halo screen. It combines three curation tracks (manual activity extraction, manual kinetics extraction, and the Issue 115 LLM pipeline) with one in-house wet lab assay.

This notebook does two things. It audits the release against every number quoted in the dry lab handoff brief, so the published figures are reproducible from the release alone rather than trusted. And it publishes the result to HuggingFace with a dataset card that documents the defects, because the release has one that blocks evaluation outright and shipping it silently would be worse than not shipping it.

The headline finding is that the split cannot evaluate. The test fold holds 7 negative labels on 5 of its 170 sequences, because the halo screen supplies 168 of the 200 negatives in the release and was allocated 178 to train, 12 to val and 0 to test. Nothing downstream is interpretable until that is re-stratified, which is Issue 117.

## Objectives

1. Verify the release on the cluster against the shipped `summary.json` and the handoff brief, re-deriving each number rather than carrying it over.
2. Quantify the two evaluation blockers: the per-fold negative deficit and the inverted label provenance.
3. Measure what a fragment remap would recover, since that gates the re-split.
4. Audit the `value` column for unit collisions, so downstream work does not regress on a mixed-unit target.
5. Publish the four tables to `petadex/petai-activity-labels` with a card that states the defects plainly.

---

## Materials and Methods

### System Initialization

Everything here is CPU-only and runs on a Killarney login node in under a second. The audit uses the standard library alone (`csv`, `collections`, `json`) so it has no dependency on the project venv. The push step needs `pandas`, `datasets` and `huggingface_hub`.

```bash
source activate_petabite.sh
export HF_HOME=/project/6122149/purav/hf_cache
```

### Data Initialization

The release was built on 2026-10-14 by the PETai curation pipeline and is not reproducible from this repo. It is treated as an input artifact. The `scp` and `unzip` commands above are the whole of its provenance on the cluster.

### Processing Steps

**Step 1: audit.** `resources/261015_issue116_petai_label_release/profile_labels.py` reads all six files and emits a JSON report covering release counts, negatives per fold, fragment recoverability, unit distribution and cluster-size asymmetry. Read-only.

```bash
python profile_labels.py /project/6122149/purav/petai-labels/combined --json audit.json
```

**Step 2: publish.** `push_to_hf.py` loads the four CSVs into typed pandas frames, pushes each as a named config, uploads `sequences.fasta` and `summary.json` under `raw/`, and writes `DATASET_CARD.md` as the repo README.

```bash
python push_to_hf.py /project/6122149/purav/petai-labels/combined \
    --repo petadex/petai-activity-labels --dry-run
python push_to_hf.py /project/6122149/purav/petai-labels/combined \
    --repo petadex/petai-activity-labels
```

### Step 0: Release shape

Confirm the six files landed intact and agree with the shipped summary before trusting anything else.

In [ ]:
import collections, csv, json
from pathlib import Path

COMBINED = Path("/project/6122149/purav/petai-labels/combined")

def read_csv(name):
    with (COMBINED / name).open(newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

labels    = read_csv("labels.csv")
sequences = read_csv("sequences.csv")
split     = read_csv("split.csv")
clusters  = read_csv("clusters.csv")
summary   = json.loads((COMBINED / "summary.json").read_text())
by_md5    = {s["seq_md5"]: s for s in sequences}

n_fasta = sum(1 for ln in (COMBINED / "sequences.fasta").open() if ln.startswith(">"))

print(f"labels.csv      {len(labels):>6,} rows   summary says {summary['labels_total']:,}")
print(f"sequences.csv   {len(sequences):>6,} rows   summary says {summary['sequences_total']:,}")
print(f"split.csv       {len(split):>6,} rows   summary says {summary['sequences_labelled_clean']:,}")
print(f"clusters.csv    {len(clusters):>6,} rows")
print(f"sequences.fasta {n_fasta:>6,} records")
print(f"distinct refs   {len({l['ref'] for l in labels if l['ref']}):>6,}")

All six agree with `summary.json`, which agrees with the handoff brief. 13,092 labels on 1,219 sequences, 1,137 of them clean, from 696 references.

### Step 1: the split cannot evaluate

Negatives are the scarce class at 1.53% of the release, so where they land decides whether any classification metric means anything. Count them per fold, and count the halo screen separately since it supplies most of them.

In [ ]:
TRUE = {"true", "1", "yes", "t"}

neg_labels = collections.Counter()
neg_seqs   = collections.defaultdict(set)
halo       = collections.Counter()
fold_seqs  = collections.defaultdict(set)

for row in labels:
    seq = by_md5.get(row["seq_md5"])
    if seq is None:
        continue
    fold = seq["split"] or "DROPPED"
    fold_seqs[fold].add(row["seq_md5"])
    if row["is_negative"].strip().lower() in TRUE:
        neg_labels[fold] += 1
        neg_seqs[fold].add(row["seq_md5"])
    if row["source"] == "halo":
        halo[fold] += 1

print(f"{'fold':<9}{'seqs':>7}{'neg labels':>13}{'neg seqs':>11}{'halo labels':>14}")
for fold in ("train", "val", "test", "DROPPED"):
    print(f"{fold:<9}{len(fold_seqs[fold]):>7}{neg_labels[fold]:>13}"
          f"{len(neg_seqs[fold]):>11}{halo[fold]:>14}")

The test fold has **7 negative labels on 5 of 170 sequences**. The halo screen went 178 train, 12 val, **0 test**.

A 170-sequence test fold with 5 inactives cannot support a precision-recall curve, a calibrated threshold, or an enrichment estimate with a usable confidence interval. This is the blocker, and it is upstream of every modelling decision.

Now the second defect, which compounds it.

In [ ]:
prov = summary["split_by_provenance"]
print(f"{'fold':<7}{'human':>8}{'llm':>7}{'mixed':>8}{'llm %':>9}")
for fold in ("train", "val", "test"):
    d = prov[fold]
    tot = sum(d.values())
    print(f"{fold:<7}{d.get('human',0):>8}{d.get('llm',0):>7}"
          f"{d.get('mixed',0):>8}{100*d.get('llm',0)/tot:>8.1f}%")

llm_neg = sum(1 for l in labels
              if l["source"] == "llm" and l["is_negative"].strip().lower() in TRUE)
print(f"\nnegatives among the {sum(1 for l in labels if l['source']=='llm'):,} LLM labels: {llm_neg}")

Provenance is inverted. Train is 19.6% LLM, test is 62.9%, and the LLM subset contains **zero** negatives across all 4,898 labels. The model learns from human-curated measurements and is scored on unvalidated machine extractions that cannot, by construction, contain a negative.

That zero is worth dwelling on. It is not a property of the enzymes. Published papers report what worked, and an extraction pipeline reading those papers inherits the bias. It means the LLM track is unusable as an evaluation set regardless of how the folds are drawn.

### Step 2: what a fragment remap recovers

82 sequences carry no split assignment because they are fragments or contained duplicates. Their labels are currently discarded. Most name a parent that is already in a fold.

In [ ]:
dropped        = [m for m, s in by_md5.items() if not s["split"]]
labels_per_seq = collections.Counter(l["seq_md5"] for l in labels)
remappable     = [m for m in dropped
                  if by_md5.get(by_md5[m]["contained_in"], {}).get("split")]

orphaned    = sum(labels_per_seq[m] for m in dropped)
recoverable = sum(labels_per_seq[m] for m in remappable)

print(f"sequences with no split      {len(dropped):>6}")
print(f"  with a parent in a fold    {len(remappable):>6}")
print(f"  genuinely unrecoverable    {len(dropped)-len(remappable):>6}")
print(f"\nlabels stranded on them      {orphaned:>6}  ({100*orphaned/len(labels):.2f}% of release)")
print(f"  recoverable by remap       {recoverable:>6}")
print(f"  lost                       {orphaned-recoverable:>6}")

frag = [l for l in labels if l["seq_md5"] in dropped]
melt_all  = [l for l in labels if "melt" in l["variable"].lower()]
melt_frag = [l for l in frag   if "melt" in l["variable"].lower()]
print(f"\nof the stranded labels: {sum(1 for l in frag if l['substrate']=='PET')} are PET measurements")
print(f"                        {len(melt_frag)} of {len(melt_all)} melting labels "
      f"({100*len(melt_frag)/len(melt_all):.1f}%)")

73 of 82 are remappable, recovering **3,150 of 3,188** stranded labels. Only 9 sequences with 38 labels are unrecoverable.

This is a quarter of the release sitting idle behind a one-line join, and it carries 1,301 PET measurements and 85.5% of all melting data. It has to happen before the re-split rather than after, because moving 3,150 labels changes the per-cluster counts that the stratification balances on.

### Step 3: `value` is not one regression target

In [ ]:
units = collections.Counter(l["unit"] for l in labels)
print("top units")
for u, n in units.most_common(12):
    print(f"  {u!r:<32}{n:>6}")

vals = [float(l["value"]) for l in labels if l["value"] not in ("", "NA")]
print(f"\nvalue spans {min(vals):,.2f} to {max(vals):,.0f} across {len(vals):,} rows")

rel  = sum(units[u] for u in ("%", "percent", "% relative activity"))
melt = collections.Counter(l["unit"] for l in melt_all)
print(f"\nrelative activity, three spellings of one quantity: {rel}")
print(f"melting labels ({len(melt_all)}) by unit: {dict(melt)}")
print(f"kinetics as three distinct targets: "
      f"kcat {sum(1 for l in labels if l['variable']=='kcat')}, "
      f"Km {sum(1 for l in labels if l['variable']=='Km')}, "
      f"kcat/Km {sum(1 for l in labels if l['variable']=='kcat/Km')}")

Nine orders of magnitude and a negative floor, because the column holds µM alongside 1/s/M alongside kcal/mol. Three collisions need resolving before any regression:

- Relative activity is spelled `%`, `percent` and `% relative activity`, 2,656 labels across the three.
- Melting data mixes 751 CD spectroscopy curve points (`fraction of folded protein`) with 151 actual Tm values in `°C` or `K`. Only the latter are temperatures.
- `kcat`, `Km` and `kcat/Km` are three targets, not one.

The clean choices are binary halo activity for classification and PET monomer release for regression.

### Step 4: cluster-size asymmetry

The reason confidence intervals have to resample clusters rather than sequences.

In [ ]:
per_fold = collections.defaultdict(collections.Counter)
for s in sequences:
    if s["split"]:
        per_fold[s["split"]][s["cluster_30"]] += 1

print(f"{'fold':<7}{'seqs':>6}{'clusters':>10}{'mean':>8}{'max':>6}{'top3 %':>9}")
for fold in ("train", "val", "test"):
    sizes = sorted(per_fold[fold].values(), reverse=True)
    print(f"{fold:<7}{sum(sizes):>6}{len(sizes):>10}{sum(sizes)/len(sizes):>8.2f}"
          f"{sizes[0]:>6}{100*sum(sizes[:3])/sum(sizes):>8.2f}%")

Train holds 70 clusters averaging 11.4 sequences with a single cluster of 124, and its top three clusters are 43.84% of the fold. Val and test hold 63 clusters each averaging 2.7.

Effective training diversity is nearer **70 families than 796 sequences**. Any sequence-level bootstrap will report a confidence interval several times too narrow.

### Step 5: publish

Dry run first, which builds every config and prints the resolved schema without contacting the Hub.

In [ ]:
!python ../resources/261015_issue116_petai_label_release/push_to_hf.py \
    /project/6122149/purav/petai-labels/combined \
    --repo petadex/petai-activity-labels --dry-run

In [ ]:
!python ../resources/261015_issue116_petai_label_release/push_to_hf.py \
    /project/6122149/purav/petai-labels/combined \
    --repo petadex/petai-activity-labels

## Results & Discussion

The release verifies exactly. Every figure in the dry lab handoff brief re-derives from the six files, with one correction noted below.

**Key metric**: the shipped split is **unusable for evaluation**. 7 negative labels on 5 of 170 test sequences.

**Release shape**:
- 13,092 labels on 1,219 sequences, 1,137 clean, 696 references
- Sources: activity 6,655, llm 4,898, kinetics 1,342, halo 197
- Negatives: 200 (1.53%), of which 168 from the halo screen
- Conditions: pH on 53.25%, temperature on 29.74%, both on 3,398 (25.95%)
- 30%-identity split is homology-clean, leakage check passes

**Observations**:

- The negative deficit is a stratification bug, not a data shortage. 200 negatives exist and 163 of the sequences carrying them are in train. The halo screen went 178/12/0 because it was split on sequence identity without balancing on source, so the one assay that produces inactives concentrated almost entirely in one fold.
- Provenance inversion is the subtler problem and may be worse. Train is 19.6% LLM, test 62.9%, and the 4,898 LLM labels contain zero negatives. A publication-derived extraction pipeline cannot produce negatives, because papers report successes. Any evaluation on the current test fold is largely an evaluation against machine-extracted positives.
- The fragment remap is the cheapest win available. 73 of 82 dropped sequences have a parent already in a fold, recovering 3,150 labels including 1,301 PET measurements and 85.5% of all melting data. It must precede the re-split.
- `value` spans -372.91 to 1.21e9. It is a single float column holding µM, 1/s/M, kcal/mol, °C and CD curve fractions. Nothing should regress on it unfiltered.
- Train is 70 clusters, not 796 sequences. The top three hold 43.84%. This caps how much a larger architecture can buy and argues for starting with few auxiliary tracks and ablating each one.

**Correction to the handoff brief**: the brief records `value` as spanning 0 to 329.8. Measured across all 13,092 rows it is -372.91 to 1.21e9. The brief's figure appears to describe the melting subset. The card uses the measured range.

**Published**: `petadex/petai-activity-labels`, public, four configs (`labels`, `sequences`, `split`, `clusters`) plus `raw/sequences.fasta` and `raw/summary.json`. The card leads with the split defect rather than burying it, documents the unit collisions, states the 32.90% MD5 overlap with PETadex and the 21 Logan hits, and flags licensing as unsettled.

**Follow-up questions**:

- Does re-stratifying on halo membership and provenance leave a test fold with enough inactives to calibrate, or does the halo screen need extending to more sequences first?
- Should LLM labels be held out of evaluation entirely and reported as training-only augmentation, given that they cannot contain negatives?
- What is the real homology overlap with PETadex? 32.90% is exact MD5 on a set full of engineered point variants. An MMseqs2 search against the 22,235 30%-identity centroids is the number that bears on transfer.
- Is the licensing resolvable before the 2026-10-21 wiki freeze, or does the card carry "unsettled" into the Jamboree?

**Next**: Issue 117, fragment remap and re-stratified split, then the ablation ladder on fixed folds.